In [0]:
%sql
use catalog deltalake_catalog;

In [0]:
landing_zone = "/Volumes/deltalake_catalog/default/raw"
orders_data = landing_zone + "/ordershistory"
checkpoint_path = landing_zone + "/orders_checkpoint"

In [0]:
ordersdf = (spark.readStream 
.format("cloudFiles") 
.option("cloudFiles.format", "csv") 
.option("cloudFiles.inferSchema", "true") 
.option("cloudFiles.inferColumnTypes", "true") 
.option("cloudFiles.schemaLocation", checkpoint_path) 
.load(orders_data))



In [0]:
%sql
drop table if exists deltalake_catalog.default.ordersdelta;

In [0]:
ordersdf.writeStream \
.format("delta") \
.option("checkpointLocation", checkpoint_path) \
.option("mergeSchema", True) \
.outputMode("append") \
.trigger(processingTime="10 seconds") \
.toTable("deltalake_catalog.default.ordersdelta")



In [0]:
%sql
select * from deltalake_catalog.default.ordersdelta;